In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from scipy.spatial import cKDTree


# ============================================================
# SETTINGS
# ============================================================

RAW_DATA_FILE = "director_raw.npz"
INTERPOLATED_DATA_FILE = "interpolated_Director_Lx400_Ly400_Lz400.npz"
HELIX_FILE = "helix.png"
OUTPUT_FOLDER = "Director_GIFs"

CENTER_XY = (200,200)
Z_CENTER = 200

TILT_ANGLES = (31,59)
ANGLE_START = 0
ANGLE_END = 180
ANGLE_STEP = 1

Z_START = 100
Z_END = 300
Z_STEP = 1

PLANE_RANGE = (-100,100)
INSET_PLANE_RANGE = (-60,60)
GRID_RESOLUTION = 300
MAX_DISTANCE = 1.0

DIRECTOR_SKIP = 12
DIRECTOR_SCALE = 40
DIRECTOR_WIDTH = 0.006

CARTESIAN_START = 140
CARTESIAN_END = 260
CARTESIAN_STEP = 1
CARTESIAN_RANGE = (140,260)
CARTESIAN_SKIP = 4
CARTESIAN_SCALE = 40
CARTESIAN_WIDTH = 0.006

COLORMAP = "twilight"
FPS = 10
DPI = 120

os.makedirs(OUTPUT_FOLDER,exist_ok=True)


# ============================================================
# LOAD RAW DIRECTOR DATA
# ============================================================

raw_data = np.load(RAW_DATA_FILE)
final_points = raw_data["final_points"]
final_vectors = raw_data["final_vectors"]

print("Raw director field:")
print("final_points:",final_points.shape)
print("final_vectors:",final_vectors.shape)

tree = cKDTree(final_points)
HELIX_IMAGE = plt.imread(HELIX_FILE)


# ============================================================
# LOAD INTERPOLATED DIRECTOR DATA
# ============================================================

interpolated_data = np.load(INTERPOLATED_DATA_FILE)

nx = interpolated_data["nx"]
ny = interpolated_data["ny"]
nz = interpolated_data["nz"]

Lx = int(interpolated_data["Lx"])
Ly = int(interpolated_data["Ly"])
Lz = int(interpolated_data["Lz"])

X,Y,Z = np.meshgrid(np.arange(Lx),np.arange(Ly),np.arange(Lz),indexing="ij")

print("\nInterpolated director field:")
print("nx:",nx.shape)
print("ny:",ny.shape)
print("nz:",nz.shape)


# ============================================================
# COMMON PLOT FUNCTIONS
# ============================================================

def add_color_wheel(ax,size=0.10,fontsize=12,colormap=COLORMAP):
    wheel_ax = ax.inset_axes([0.92-size,0.92-size,size,size],projection="polar")

    t = np.linspace(0,2*np.pi,361)
    r = np.linspace(0.5,1.0,20)
    T,R = np.meshgrid(t,r)

    wheel_ax.pcolormesh(T,R,np.mod(T,np.pi),cmap=colormap,vmin=0,vmax=np.pi,shading="auto")
    wheel_ax.set_facecolor("black")
    wheel_ax.set_yticks([])
    wheel_ax.set_xticks([0,np.pi/2,np.pi,3*np.pi/2])
    wheel_ax.set_xticklabels(["0°","90°","0°","90°"])
    wheel_ax.tick_params(axis="x",colors="white",labelsize=fontsize,pad=6)

    for label in wheel_ax.get_xticklabels():
        label.set_fontweight("bold")

    wheel_ax.spines["polar"].set_color("white")
    wheel_ax.spines["polar"].set_linewidth(1.5)
    wheel_ax.grid(False)


def format_axes_black(ax,xlim,ylim,xlabel="",ylabel="",show_ticks=False,fontsize=16):
    ax.set_facecolor("black")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal")
    ax.grid(False)

    if show_ticks:
        ax.set_xlabel(xlabel,color="white",fontsize=fontsize,fontweight="bold")
        ax.set_ylabel(ylabel,color="white",fontsize=fontsize,fontweight="bold")
        ax.tick_params(axis="both",colors="white",labelsize=fontsize-4,width=2)

        for label in ax.get_xticklabels()+ax.get_yticklabels():
            label.set_fontweight("bold")
    else:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel("")
        ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_color("white")
        spine.set_linewidth(2)


def get_plane_basis(angle_deg):
    angle = np.deg2rad(angle_deg)

    normal = np.array([np.sin(angle),0.0,np.cos(angle)])
    e1 = np.array([np.cos(angle),0.0,-np.sin(angle)])
    e2 = np.array([0.0,1.0,0.0])

    return normal,e1,e2


# ============================================================
# RAW-FIELD PLANE INTERPOLATION
# ============================================================

def interpolate_tilted_plane(center,e1,e2,plane_range=PLANE_RANGE,grid_resolution=GRID_RESOLUTION,max_distance=MAX_DISTANCE,tol=1e-8):
    pmin,pmax = plane_range

    u = np.linspace(pmin,pmax,grid_resolution)
    v = np.linspace(pmin,pmax,grid_resolution)
    U,V = np.meshgrid(u,v)

    points = center[None,None,:]+U[:,:,None]*e1[None,None,:]+V[:,:,None]*e2[None,None,:]
    distances,indices = tree.query(points.reshape(-1,3),k=1)

    vectors = final_vectors[indices].copy()

    nU = vectors@e1
    nV = vectors@e2

    projection = np.sqrt(nU**2+nV**2)
    theta = np.mod(np.arctan2(nV,nU),np.pi)

    invalid = (distances>max_distance)|(projection<tol)

    theta[invalid] = np.nan
    nU[invalid] = np.nan
    nV[invalid] = np.nan

    return U,V,theta.reshape(U.shape),nU.reshape(U.shape),nV.reshape(U.shape)


def add_director_rods(ax,U,V,nU,nV,k=DIRECTOR_SKIP,scale=DIRECTOR_SCALE,width=DIRECTOR_WIDTH):
    Up = U[::k,::k]
    Vp = V[::k,::k]
    nUp = nU[::k,::k]
    nVp = nV[::k,::k]

    valid = np.isfinite(nUp)&np.isfinite(nVp)

    ax.quiver(Up[valid],Vp[valid],nUp[valid],nVp[valid],color="black",pivot="middle",scale=scale,width=width,headaxislength=0,headlength=0,headwidth=1,linewidths=width)


# ============================================================
# 3D INSET
# ============================================================

def add_3d_inset(ax,center,normal,e1,e2):
    pmin,pmax = INSET_PLANE_RANGE

    u = np.linspace(pmin,pmax,30)
    v = np.linspace(pmin,pmax,30)
    U,V = np.meshgrid(u,v)

    X_plane = center[0]+U*e1[0]+V*e2[0]
    Y_plane = center[1]+U*e1[1]+V*e2[1]
    Z_plane = center[2]+U*e1[2]+V*e2[2]

    ax3d = ax.inset_axes([0.0,0.78,0.22,0.22],projection="3d")
    ax3d.set_facecolor("white")

    # Remove grid and axis outline
    ax3d.grid(False)
    ax3d.xaxis.line.set_color((1,1,1,0))
    ax3d.yaxis.line.set_color((1,1,1,0))
    ax3d.zaxis.line.set_color((1,1,1,0))
    ax3d.xaxis._axinfo["grid"]["linewidth"] = 0
    ax3d.yaxis._axinfo["grid"]["linewidth"] = 0
    ax3d.zaxis._axinfo["grid"]["linewidth"] = 0

    # Tilted plane
    ax3d.plot_surface(X_plane,Y_plane,Z_plane,alpha=0.35)

    # Plane normal
    ax3d.quiver(center[0],center[1],center[2],normal[0]*40,normal[1]*40,normal[2]*40,linewidth=2,color="black")

    # Helix image
    imagebox = OffsetImage(HELIX_IMAGE,zoom=0.25)
    ax3d.add_artist(AnnotationBbox(imagebox,(0.5,0.5),xycoords="axes fraction",frameon=False))

    # Center
    ax3d.scatter(center[0],center[1],center[2],s=10,color="black")

    ax3d.set_xlim(130,270)
    ax3d.set_ylim(130,270)
    ax3d.set_zlim(120,280)
    ax3d.set_box_aspect((1,1,1))

    ax3d.set_xticks([])
    ax3d.set_yticks([])
    ax3d.set_zticks([])
    ax3d.set_xlabel("")
    ax3d.set_ylabel("")
    ax3d.set_zlabel("")

    # XYZ labels
    ax3d.text2D(0.94,0.02,r"$x$",transform=ax3d.transAxes,color="black",fontsize=18,ha="right",va="bottom")
    ax3d.text2D(0.12,0.02,r"$y$",transform=ax3d.transAxes,color="black",fontsize=18,ha="left",va="bottom")
    ax3d.text2D(0.12,0.82,r"$z$",transform=ax3d.transAxes,color="black",fontsize=18,ha="left",va="center")


# ============================================================
# ROTATING-PLANE GIF
# ============================================================

def make_rotating_plane_gif(show_directors=False,filename="director_rotating_plane.gif"):
    pmin,pmax = PLANE_RANGE

    cmap = plt.get_cmap(COLORMAP).copy()
    cmap.set_bad("black")

    fig = plt.figure(figsize=(8,8))
    fig.patch.set_facecolor("black")

    center = np.array([CENTER_XY[0],CENTER_XY[1],Z_CENTER],dtype=float)

    def update(angle_deg):
        fig.clear()
        fig.patch.set_facecolor("black")

        ax = fig.add_axes([0.10,0.10,0.80,0.80])
        ax.set_facecolor("black")

        normal,e1,e2 = get_plane_basis(angle_deg)
        U,V,theta,nU,nV = interpolate_tilted_plane(center,e1,e2)

        ax.pcolormesh(U,V,theta,cmap=cmap,vmin=0,vmax=np.pi,shading="auto")

        if show_directors:
            add_director_rods(ax,U,V,nU,nV)

        format_axes_black(ax,(pmin,pmax),(pmin,pmax))

        if show_directors:
            title = f"Director orientation in rotating plane with vector field\nangle = {angle_deg:.0f}°"
        else:
            title = f"Director orientation in rotating plane\nangle = {angle_deg:.0f}°"

        ax.set_title(title,color="white",fontsize=15,fontweight="bold",pad=15)

        add_color_wheel(ax)
        add_3d_inset(ax,center,normal,e1,e2)

    frames = np.arange(ANGLE_START,ANGLE_END+ANGLE_STEP,ANGLE_STEP)

    ani = FuncAnimation(fig,update,frames=frames,interval=1000/FPS,repeat=True)
    ani.save(filename,writer=PillowWriter(fps=FPS),dpi=DPI,savefig_kwargs={"facecolor":"black"})

    plt.close(fig)
    print("Saved:",filename)


# ============================================================
# TILTED-PLANE Z-SWEEP GIF
# ============================================================

def make_tilted_plane_z_sweep_gif(angle_deg,show_directors=False,filename=None):
    if filename is None:
        suffix = "with_vectors" if show_directors else "no_vectors"
        filename = f"director_tilted_plane_{angle_deg}deg_z_sweep_{suffix}.gif"

    normal,e1,e2 = get_plane_basis(angle_deg)
    pmin,pmax = PLANE_RANGE

    cmap = plt.get_cmap(COLORMAP).copy()
    cmap.set_bad("black")

    fig,ax = plt.subplots(figsize=(8,8))
    fig.patch.set_facecolor("black")

    def update(z_center):
        ax.clear()
        ax.set_facecolor("black")

        center = np.array([CENTER_XY[0],CENTER_XY[1],z_center],dtype=float)
        U,V,theta,nU,nV = interpolate_tilted_plane(center,e1,e2)

        ax.pcolormesh(U,V,theta,cmap=cmap,vmin=0,vmax=np.pi,shading="auto")

        if show_directors:
            add_director_rods(ax,U,V,nU,nV)

        format_axes_black(ax,(pmin,pmax),(pmin,pmax))

        if show_directors:
            title = f"Director orientation in tilted plane with vector field\nangle = {angle_deg}°, center Z = {z_center}"
        else:
            title = f"Director orientation in tilted plane\nangle = {angle_deg}°, center Z = {z_center}"

        ax.set_title(title,color="white",fontsize=15,fontweight="bold",pad=15)
        add_color_wheel(ax)

    frames = np.arange(Z_START,Z_END+1,Z_STEP)

    ani = FuncAnimation(fig,update,frames=frames,interval=1000/FPS,repeat=True)
    ani.save(filename,writer=PillowWriter(fps=FPS),dpi=DPI,savefig_kwargs={"facecolor":"black"})

    plt.close(fig)
    print("Saved:",filename)


# ============================================================
# 3D ROTATING-PLANE CHECK GIF
# ============================================================

def make_rotating_plane_3d_gif(filename="director_rotating_plane_3d.gif"):
    center = np.array([200,200,200],dtype=float)

    pmin,pmax = INSET_PLANE_RANGE
    u = np.linspace(pmin,pmax,50)
    v = np.linspace(pmin,pmax,50)
    U,V = np.meshgrid(u,v)

    fig = plt.figure(figsize=(10,10))
    ax = fig.add_subplot(111,projection="3d")

    def update(angle_deg):
        ax.clear()

        normal,e1,e2 = get_plane_basis(angle_deg)

        X_plane = center[0]+U*e1[0]+V*e2[0]
        Y_plane = center[1]+U*e1[1]+V*e2[1]
        Z_plane = center[2]+U*e1[2]+V*e2[2]

        ax.plot_surface(X_plane,Y_plane,Z_plane,alpha=0.35)
        ax.quiver(center[0],center[1],center[2],normal[0]*40,normal[1]*40,normal[2]*40,linewidth=3,color="black")

        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")

        ax.set_xlim(130,270)
        ax.set_ylim(130,270)
        ax.set_zlim(120,280)
        ax.set_box_aspect((1,1,1))

        ax.set_title(f"Plane tilted {angle_deg}° from the Z axis")

    frames = np.arange(ANGLE_START,ANGLE_END+ANGLE_STEP,ANGLE_STEP)

    ani = FuncAnimation(fig,update,frames=frames,interval=1000/FPS,repeat=True)
    ani.save(filename,writer=PillowWriter(fps=FPS),dpi=DPI)

    plt.close(fig)
    print("Saved:",filename)


# ============================================================
# CARTESIAN X-Y Z-SWEEP GIF
# ============================================================

def make_xy_z_sweep_gif(filename="director_xy_z_sweep_with_vectors.gif"):
    cmap = plt.get_cmap(COLORMAP).copy()
    cmap.set_bad("black")

    fig,ax = plt.subplots(figsize=(8,8))
    fig.patch.set_facecolor("black")

    def update(z):
        ax.clear()
        ax.set_facecolor("black")

        nx_full = nx[:,:,z]
        ny_full = ny[:,:,z]

        projection = np.sqrt(nx_full**2+ny_full**2)
        theta = np.ma.array(np.mod(np.arctan2(ny_full,nx_full),np.pi),mask=projection<1e-8)

        ax.pcolormesh(X[:,:,z],Y[:,:,z],theta,cmap=cmap,vmin=0,vmax=np.pi,shading="nearest")

        Xp = X[::CARTESIAN_SKIP,::CARTESIAN_SKIP,z]
        Yp = Y[::CARTESIAN_SKIP,::CARTESIAN_SKIP,z]
        nxp = nx[::CARTESIAN_SKIP,::CARTESIAN_SKIP,z]
        nyp = ny[::CARTESIAN_SKIP,::CARTESIAN_SKIP,z]

        valid = np.sqrt(nxp**2+nyp**2)>=1e-8

        ax.quiver(Xp[valid],Yp[valid],nxp[valid],nyp[valid],color="black",pivot="middle",scale=CARTESIAN_SCALE,width=CARTESIAN_WIDTH,headaxislength=0,headlength=0,headwidth=1,linewidths=CARTESIAN_WIDTH)

        format_axes_black(ax,CARTESIAN_RANGE,CARTESIAN_RANGE,"X","Y",show_ticks=True)

        ax.set_title(f"Director orientation in X-Y plane at Z = {z}",color="white",fontsize=15,fontweight="bold",pad=15)

        add_color_wheel(ax)

    frames = np.arange(CARTESIAN_START,CARTESIAN_END+1,CARTESIAN_STEP)

    ani = FuncAnimation(fig,update,frames=frames,interval=1000/FPS,repeat=True)
    ani.save(filename,writer=PillowWriter(fps=FPS),dpi=DPI,savefig_kwargs={"facecolor":"black"})

    plt.close(fig)
    print("Saved:",filename)


# ============================================================
# CARTESIAN Y-Z X-SWEEP GIF
# ============================================================

def make_yz_x_sweep_gif(filename="director_yz_x_sweep_with_vectors.gif"):
    cmap = plt.get_cmap(COLORMAP).copy()
    cmap.set_bad("black")

    fig,ax = plt.subplots(figsize=(8,8))
    fig.patch.set_facecolor("black")

    def update(x):
        ax.clear()
        ax.set_facecolor("black")

        ny_full = ny[x,:,:]
        nz_full = nz[x,:,:]

        projection = np.sqrt(ny_full**2+nz_full**2)
        theta = np.ma.array(np.mod(np.arctan2(nz_full,ny_full),np.pi),mask=projection<1e-8)

        ax.pcolormesh(Y[x,:,:],Z[x,:,:],theta,cmap=cmap,vmin=0,vmax=np.pi,shading="nearest")

        Yp = Y[x,::CARTESIAN_SKIP,::CARTESIAN_SKIP]
        Zp = Z[x,::CARTESIAN_SKIP,::CARTESIAN_SKIP]
        nyp = ny[x,::CARTESIAN_SKIP,::CARTESIAN_SKIP]
        nzp = nz[x,::CARTESIAN_SKIP,::CARTESIAN_SKIP]

        valid = np.sqrt(nyp**2+nzp**2)>=1e-8

        ax.quiver(Yp[valid],Zp[valid],nyp[valid],nzp[valid],color="black",pivot="middle",scale=CARTESIAN_SCALE,width=CARTESIAN_WIDTH,headaxislength=0,headlength=0,headwidth=1,linewidths=CARTESIAN_WIDTH)

        format_axes_black(ax,CARTESIAN_RANGE,CARTESIAN_RANGE,"Y","Z",show_ticks=True)

        ax.set_title(f"Director orientation in Y-Z plane at X = {x}",color="white",fontsize=15,fontweight="bold",pad=15)

        add_color_wheel(ax)

    frames = np.arange(CARTESIAN_START,CARTESIAN_END+1,CARTESIAN_STEP)

    ani = FuncAnimation(fig,update,frames=frames,interval=1000/FPS,repeat=True)
    ani.save(filename,writer=PillowWriter(fps=FPS),dpi=DPI,savefig_kwargs={"facecolor":"black"})

    plt.close(fig)
    print("Saved:",filename)


# ============================================================
# CREATE ALL GIFS
# ============================================================

print("\nCreating GIFs...\n")

# Rotating plane
make_rotating_plane_gif(
    show_directors=False,
    filename=os.path.join(OUTPUT_FOLDER,"director_rotating_plane_no_vectors.gif")
)

make_rotating_plane_gif(
    show_directors=True,
    filename=os.path.join(OUTPUT_FOLDER,"director_rotating_plane_with_vectors.gif")
)

# Tilted planes at 31° and 59°
for angle_deg in TILT_ANGLES:
    make_tilted_plane_z_sweep_gif(
        angle_deg=angle_deg,
        show_directors=False,
        filename=os.path.join(OUTPUT_FOLDER,f"director_tilted_plane_{angle_deg}deg_z_sweep_no_vectors.gif")
    )

    make_tilted_plane_z_sweep_gif(
        angle_deg=angle_deg,
        show_directors=True,
        filename=os.path.join(OUTPUT_FOLDER,f"director_tilted_plane_{angle_deg}deg_z_sweep_with_vectors.gif")
    )

# 3D rotating-plane check
make_rotating_plane_3d_gif(
    filename=os.path.join(OUTPUT_FOLDER,"director_rotating_plane_3d.gif")
)

# Cartesian planes
make_xy_z_sweep_gif(
    filename=os.path.join(OUTPUT_FOLDER,"director_xy_z_sweep_with_vectors.gif")
)

make_yz_x_sweep_gif(
    filename=os.path.join(OUTPUT_FOLDER,"director_yz_x_sweep_with_vectors.gif")
)

print("\nAll GIFs finished.")

Raw director field:
final_points: (11766000, 3)
final_vectors: (11766000, 3)

Interpolated director field:
nx: (400, 400, 400)
ny: (400, 400, 400)
nz: (400, 400, 400)

Creating GIFs...

Saved: Director_GIFs/director_rotating_plane_no_vectors.gif
Saved: Director_GIFs/director_rotating_plane_with_vectors.gif
Saved: Director_GIFs/director_tilted_plane_31deg_z_sweep_no_vectors.gif
Saved: Director_GIFs/director_tilted_plane_31deg_z_sweep_with_vectors.gif
Saved: Director_GIFs/director_tilted_plane_59deg_z_sweep_no_vectors.gif
Saved: Director_GIFs/director_tilted_plane_59deg_z_sweep_with_vectors.gif
Saved: Director_GIFs/director_rotating_plane_3d.gif
Saved: Director_GIFs/director_xy_z_sweep_with_vectors.gif
Saved: Director_GIFs/director_yz_x_sweep_with_vectors.gif

All GIFs finished.
